## Testdaten
Für das Projekt müssen zumindest einmalig Testdaten generiert werden, die zum RAG-Datensatz passen. Das werde ich wieder mit einem LLM erstellen lassen, ich denke, die Aufgabe ist nicht schwer wenn es nur dedizierte Daten erhält. Die Testmenge wird überschaubar bleiben und kann in handarbeit evaluiert werden.

In [21]:
import os
import json
import pandas as pd
from mistralai import Mistral
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
model = 'mistral-medium-2508'
client = Mistral(api_key=api_key, timeout_ms=120000)

# Requestfunktion
def agent_request(system_promt, schema, content):

    response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'system',
                'content': system_promt
            },
            {
                'role': 'user',
                'content': content,
            }
        ],
        response_format = {
            "type": "json_object",
            "json_schema": schema
        }
    )

    return response

# Daten laden
df = pd.read_json('../data/processed/products_chunked.jsonl', lines=True)


## Dataprep

Es soll zwei Datensätze an Testdaten geben. Zum eine Fragen, für deren Beantwortung ein bestimmter Chunk gefunden werden muss, zum anderen solche, deren Antworten über mehrere Chunks verteilt ist.

Für die Singe-Chunk-Fragen werden nur technische Daten verwendet. Um diese filtern zu können muss auf die Metadaten zugegriffen werden, die ein JSON-Objekt mit den Daten enthalten. Eine Möglichkeit ist, den Typen in eine eigene Spalte zu schreiben, ist letztlich ja pro Chunk typisch.

In [26]:
df = pd.read_json('../data/processed/products_chunked.jsonl', lines=True)
df['chunk_type'] = df['metadata'].apply(pd.Series)['chunk_type']

specs_df = df[df['chunk_type'] == 'spec']
descs_df = df[df['chunk_type'] == 'desc']

print(specs_df)

                                                     id  \
5     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
6     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
7     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
8     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
9     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
...                                                 ...   
5288  SRPvg-1402-Performance-Laborkuehlgeraet-mit-Um...   
5289  SRPvg-1402-Performance-Laborkuehlgeraet-mit-Um...   
5290  SRPvg-1402-Performance-Laborkuehlgeraet-mit-Um...   
5291  SRPvg-1402-Performance-Laborkuehlgeraet-mit-Um...   
5292  SRPvg-1402-Performance-Laborkuehlgeraet-mit-Um...   

                                               document  \
5     Der Kirsch LABO-288 PRO-ACTIVE hat die Außenma...   
6     Bei 90 Grad geöffneter Tür hat der Kirsch LABO...   
7     Der Kirsch LABO-288 PRO-ACTIVE hat die Innenma...   
8     Der Kirsch LABO-288 PRO-ACTIVE hat einen Kühli...

## Generation

Aus den gefilterten Datensätzen sollen zufällig Datensätze gezogen und an das LLM zur Generierung der Testfragen gesendet werden.